In [1]:
import pandas as pd
import numpy as np
import ast
import os
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from typing import Any

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.1) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


# 2. Get the embedding of each PSOC task

In [2]:
# Open up the mca
mca_df = pd.read_csv('../data/auxiliary/mca_soc_codes.csv', dtype={'PSOC Code':str, 'ISCO Code':str})
mca_df['SOC Codes'] = mca_df['2019 SOC Codes'].apply(ast.literal_eval)

In [27]:
def create_mapping(
    df_filepath: str,
    group_column: str,
    value_column: str,
    is_numeric=True
) -> dict[str, np.ndarray]:
    """
    Create a mapping from each group value to its corresponding row values.

    Each unique value in `group_column` becomes a key in the returned
    dictionary. The corresponding value is a NumPy array containing the
    values from `value_column` for rows belonging to that group.

    Args:
        df_filepath: Path to the CSV file containing the data.
        group_column: Column used to group the rows.
        value_column: Column containing the values to collect.

    Returns:
        A dictionary mapping each group value to a NumPy array of
        corresponding values from `value_column`.
    """
    # open the data
    df = pd.read_csv(df_filepath)
    if is_numeric:
        df[value_column] = df[value_column].apply(ast.literal_eval)

    # create the mapping
    if is_numeric:
        return {
            str(group_value): np.array(group_values.tolist(), dtype=np.float32)
            for group_value, group_values in df.groupby(group_column)[value_column]
        }
    else:
        return {
            str(group_value): group_values.tolist()
            for group_value, group_values in df.groupby(group_column)[value_column]
        }

In [4]:
# Create mappings for MCA and SOC title embeddings
embedded_mca_title_map = create_mapping(
    "../data/auxiliary/embedded_mca_title.csv",
    "Job Title",
    "Job Title Embedded",
)
embedded_soc_title_map = create_mapping(
    "../data/auxiliary/embedded_soc_title.csv",
    "O*NET-SOC Code",
    "Job Title Embedded",
)

# Create mappings for PSOC and SOC task embeddings
embedded_psoc_tasks_map = create_mapping(
    "../data/auxiliary/embedded_psoc_tasks.csv",
    "Code",
    "Statements Embedded",
)
embedded_soc_tasks_map = create_mapping(
    "../data/auxiliary/embedded_soc_tasks.csv",
    "O*NET-SOC Code",
    "Task Embedded",
)

In [5]:
mca_df["Job Title Embedded"] = mca_df["Job Title"].apply(
    lambda title: embedded_mca_title_map[title]
)

mca_df["SOC Titles Embedded"] = mca_df["SOC Codes"].apply(
    lambda titles: [
        embedded_soc_title_map[title]
        for title in titles
    ]
)

mca_df["PSOC Tasks Embedded"] = mca_df["PSOC Code"].apply(
    lambda task: embedded_psoc_tasks_map[task])

mca_df["SOC Tasks Embedded"] = mca_df["SOC Codes"].apply(
    lambda codes: [
        embedded_soc_tasks_map[code.strip()]
        for code in codes
    ]
)

# 2. Choose the most representative SOC code for each Occupation based on Cosine Similarity

For each PSOC occupation, we compare its tasks and job title against those of each candidate SOC code using cosine similarity between their embeddings.

For the tasks, we find the most similar SOC task for each PSOC task and take the mean of these best-match scores. Separately, we calculate the cosine similarity between the PSOC job title and the SOC title.

We then take the mean of the task similarity and title similarity to obtain the overall PSOC-SOC similarity score.

**PSOC tasks + job title -> task similarity + title similarity -> average -> representative SOC code**

A higher overall score indicates that the SOC is more semantically representative of the PSOC occupation.

In [33]:
code_titles_map = create_mapping(
    "../data/auxiliary/embedded_soc_title.csv",
    "O*NET-SOC Code",
    "Job Title",
    is_numeric=False
)

In [34]:
def estimate_representative_soc_codes(job, verbose=False):
    """
    Rank candidate SOC codes from most to least representative.

    Each candidate SOC code is evaluated using:
    1. Task similarity: mean of the best cosine similarity for each
       PSOC task against the candidate SOC's tasks.
    2. Title similarity: maximum cosine similarity between the PSOC
       job title and any of the candidate SOC's titles.

    The final score is a weighted combination of task and title similarity.

    Returns:
        Tuple containing:
        - List of SOC codes sorted by score
        - List of best-matching SOC titles corresponding to each SOC code
    """

    # If there is only one 2019 SOC code, leave it unchanged
    if len(job['2019 SOC Codes']) == 1:
        return job['2019 SOC Codes'], [
            code_titles_map[code][0]
            for code in job['2019 SOC Codes']
        ]

    embedded_psoc_tasks = np.array(
        job['PSOC Tasks Embedded']
    )

    embedded_job_title = np.array(
        job['Job Title Embedded']
    ).reshape(1, -1)

    soc_scores = []

    # Evaluate every candidate SOC code
    for embedded_soc_tasks, embedded_soc_titles, soc_code in zip(
        job['SOC Tasks Embedded'],
        job['SOC Titles Embedded'],
        job['SOC Codes']
    ):
        # -------------------------
        # Task similarity
        # -------------------------

        similarity_matrix = cosine_similarity(
            embedded_psoc_tasks,
            np.array(embedded_soc_tasks)
        )

        best_similarity_scores = similarity_matrix.max(axis=1)

        task_similarity = best_similarity_scores.mean()

        # -------------------------
        # Title similarity
        # -------------------------

        embedded_soc_titles = np.array(
            embedded_soc_titles
        )

        if embedded_soc_titles.ndim == 1:
            embedded_soc_titles = embedded_soc_titles.reshape(1, -1)

        title_similarity_matrix = cosine_similarity(
            embedded_job_title,
            embedded_soc_titles
        )

        best_title_index = title_similarity_matrix.argmax()

        title_similarity = title_similarity_matrix[
            0,
            best_title_index
        ]

        # Get actual title using the SOC code
        best_soc_title = code_titles_map[soc_code][best_title_index]

        # -------------------------
        # Final score
        # -------------------------

        score = (
            task_similarity * 0.5
            + title_similarity * 0.5
        )

        soc_scores.append({
            'SOC Code': soc_code,
            'Best SOC Title': best_soc_title,
            'Task Similarity': task_similarity,
            'Title Similarity': title_similarity,
            'Score': score
        })

    # Sort from most likely to least likely
    soc_scores = sorted(
        soc_scores,
        key=lambda x: x['Score'],
        reverse=True
    )

    result_soc_codes = [
        soc_score['SOC Code']
        for soc_score in soc_scores
    ]

    result_soc_titles = [
        soc_score['Best SOC Title']
        for soc_score in soc_scores
    ]

    if verbose:
        for rank, result in enumerate(soc_scores, start=1):
            print(
                f"{rank}. {result['SOC Code']}: "
                f"Task = {result['Task Similarity']:.4f}, "
                f"Title = {result['Title Similarity']:.4f}, "
                f"Score = {result['Score']:.4f}\n"
                f"   Best title: {result['Best SOC Title']}"
            )

    return result_soc_codes, result_soc_titles

In [ ]:
# get the most representative SOC codes
mca_df[['Sorted SOC Codes', 'Sorted SOC Titles']] = mca_df.apply(
    estimate_representative_soc_codes,
    axis=1,
    result_type='expand'
)

mca_df['SOC Code'] = mca_df['Sorted SOC Codes'].apply(
    lambda soc_codes : soc_codes[0]
)
mca_df['SOC Title'] = mca_df['Sorted SOC Titles'].apply(
    lambda soc_titles : soc_titles[0]
)

In [46]:
relevant_cols = [
    'Job Title',
    'Educational Qualification',
    'Job Sector',
    'Educational Pathway',
    'HEI with PRC Exam',
    'Some HEI', 
    'Job Subsector',
    'PSOC Code',
    'ISCO Code',
    'SOC Code',
    'SOC Title',
]

mca_df[relevant_cols].to_csv('../data/auxiliary/final_mca_soc_code.csv', index=False)

The two examples below demonstrate how accurately the system identifies the most representative SOC code. 

For example, the teachers in the first table had 37 possible SOC codes to choose from, while the occupational therapist in the second table had 19. However, the previous approach would naively assign the same general SOC code repeatedly across the different teacher occupations, which is as 25-1199.00 (Postsecondary Teachers, All Other). This results in a loss of variety and specificity in the mappings. Similarly, the occupational therapist could be assigned the general code 31-9099.00 (Healthcare Support Workers, All Other). 

With the new system, the combination of task and title similarity allows the system to distinguish between occupations and select more specific and representative SOC codes from the available candidates.

In [39]:
is_instructor = (mca_df['PSOC Code'] == '2310')
mca_df.loc[is_instructor, ['Job Title', 'SOC Code', 'SOC Title']]

,Job Title,SOC Code,SOC Title
103,Ethics Research Assistant,25-1126.00,Ethics Professor
110,Development Researcher,25-1066.00,Child Development Professor
126,Humanities Researcher,25-1069.00,Humanities Teacher
129,Legislative Researcher,25-1065.00,Public Policy Professor
197,Religious Research Assistant,25-1126.00,Religious Studies Professor
506,Anatomy and Physiology Instructor,25-1042.00,Anatomy Instructor
514,Biology Instructor,25-1042.00,Biology Instructor
645,Mathematics Instructor,25-1022.00,Mathematics Teacher
673,Philosophy and Ethics Instructor,25-1126.00,Philosophy Instructor


In [45]:
mca_df[mca_df['Job Title'] == 'AI Engineer']

,Job Title,Educational Qualification,Job Sector,Educational Pathway,HEI with PRC Exam,Some HEI,Job Subsector,PSOC Code,ISCO Code,2010 SOC Codes,...,2019 SOC Codes,SOC Codes,Job Title Embedded,SOC Titles Embedded,PSOC Tasks Embedded,SOC Tasks Embedded,Sorted SOC Codes,SOC Code,Sorted SOC Titles,SOC Title
504,AI Engineer,No Match Found,Information and Communication,Higher Education,No,No,"Computer programming, consultancy and related ...",2512,2512,"['15-1132', '15-1133']",...,"['15-1253.00', '15-1252.00']","[15-1253.00, 15-1252.00]","[[0.017561834, 0.027509617, -0.06003201, -0.01...","[[[-0.04631555, -0.017469266, -0.036145102, 0....","[[0.074845105, -0.0458888, -0.023512108, -0.03...","[[[-0.010566963, 0.0071895397, -0.02587218, 0....","[15-1252.00, 15-1253.00]",15-1252.00,[AI Specialist (Artificial Intelligence Specia...,AI Specialist (Artificial Intelligence Special...
